In [13]:
from google.colab import files
uploaded = files.upload()  # upload claims_reference.xlsx when prompted

Saving claims_reference.xlsx to claims_reference.xlsx


In [3]:
# ── Cell 1 — Setup & Generate Mechanism Terms ─────────────────────────────

import pandas as pd
import json
from google.colab import auth
from google.colab import files
import google.genai as genai

# ── Authenticate with Vertex AI ────────────────────────────────────────────
auth.authenticate_user()

client = genai.Client(
    vertexai=True,
    project='project-7f940e6f-ba1d-404b-948',
    location='global'
)

# ── Upload and load search_units ───────────────────────────────────────────
uploaded = files.upload()
search_units = pd.read_excel('search_units.xlsx')
# Parse list columns
import ast
search_units['disease_synonyms'] = search_units['disease_synonyms'].apply(ast.literal_eval)
search_units['claim_ids_covered'] = search_units['claim_ids_covered'].apply(ast.literal_eval)

print(f'Search units loaded: {len(search_units)}')
print(f'Unique clusters: {search_units["cluster_label"].nunique()}')

# ── Get unique clusters ────────────────────────────────────────────────────
clusters = search_units['cluster_label'].unique().tolist()
print(f'\nClusters to process: {clusters}')

# ── Generate mechanism terms for each cluster ──────────────────────────────
mechanism_map = {}

for cluster in clusters:
    prompt = f"""You are a biomedical expert helping build PubMed search queries.

For the biomedical cluster '{cluster}', list 6-8 biological mechanism terms
that researchers use in academic papers when studying herbal treatments for
this condition.

Rules:
- Return ONLY the terms, comma separated, nothing else
- Use standard PubMed/biomedical vocabulary
- Focus on mechanisms of action, not disease names
- Examples of good terms: anti-inflammatory, collagen synthesis,
  hepatoprotective, antimicrobial, vasodilation, antioxidant

Cluster: {cluster}
Mechanism terms:"""

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )

    terms = [t.strip() for t in response.text.strip().split(',')]
    mechanism_map[cluster] = terms
    print(f'✓ {cluster}: {terms}')

# ── Save mechanism map ─────────────────────────────────────────────────────
with open('mechanism_map.json', 'w') as f:
    json.dump(mechanism_map, f, indent=2)

print(f'\n✓ Mechanism terms generated for {len(mechanism_map)} clusters')
print('Saved to mechanism_map.json')

Saving search_units.xlsx to search_units.xlsx
Search units loaded: 5014
Unique clusters: 31

Clusters to process: ['dermatological_conditions', 'gastrointestinal_conditions', 'hair_scalp_conditions', 'metabolic_conditions', 'musculoskeletal_conditions', 'neurological_conditions', 'renal_conditions', 'respiratory_conditions', 'hepatic_conditions', 'infectious_conditions', 'reproductive_conditions', 'urological_conditions', 'wound_healing', 'oncological_conditions', 'otolaryngological_conditions', 'parasitic_conditions', 'oedema_and_fluid_conditions', 'cardiovascular_conditions', 'inflammatory_conditions', 'poisoning_and_toxicity', 'haematological_conditions', 'oral_dental_conditions', 'fever_and_systemic_illness', 'pain_management', 'ophthalmological_conditions', 'general_weakness_fatigue', 'psychological_conditions', 'humoral_conditions', 'endocrine_conditions', 'nutritional_deficiency', 'treatment_response_issues']
✓ dermatological_conditions: ['Anti-inflammatory', 'Antioxidant', 'Imm

In [4]:
# ── Cell 2 — Load & Prepare Inputs ────────────────────────────────────────

import ast
import json
import re

# ── Load mechanism map from disk ───────────────────────────────────────────
with open('mechanism_map.json', 'r') as f:
    mechanism_map = json.load(f)

# Normalize to lowercase
mechanism_map = {
    cluster: [term.lower().strip() for term in terms]
    for cluster, terms in mechanism_map.items()
}

print(f'✓ Mechanism map loaded: {len(mechanism_map)} clusters')
print(f'  Sample (wound_healing): {mechanism_map["wound_healing"]}')
print()

# ── Parse list columns in search_units ────────────────────────────────────
def safe_parse(val):
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        return ast.literal_eval(val)
    return []

search_units['disease_synonyms'] = search_units['disease_synonyms'].apply(safe_parse)
search_units['claim_ids_covered'] = search_units['claim_ids_covered'].apply(safe_parse)

# ── Verify parsing worked ──────────────────────────────────────────────────
sample_synonyms = search_units['disease_synonyms'].iloc[0]
sample_claims = search_units['claim_ids_covered'].iloc[0]

print(f'disease_synonyms type: {type(sample_synonyms)}')
print(f'claim_ids_covered type: {type(sample_claims)}')
print(f'Sample disease_synonyms (first 3): {sample_synonyms[:3]}')
print(f'Sample claim_ids_covered: {sample_claims}')

# ── Prepare other_known_names as a lookup dict ─────────────────────────────
# For each search unit, split other_known_names into a clean list
def parse_other_names(raw):
    if pd.isna(raw) or str(raw).strip() == '':
        return []
    return [n.strip() for n in str(raw).split(',') if len(n.strip()) > 3]

search_units['other_names_list'] = search_units['other_known_names'].apply(parse_other_names)

print(f'✓ other_known_names parsed')
print(f'  Sample (entry_id=1): {search_units[search_units["entry_id"]==1]["other_names_list"].iloc[0]}')
print()

# ── Final confirmation ─────────────────────────────────────────────────────
print(f'Search units ready: {len(search_units)}')
print(f'Columns: {search_units.columns.tolist()}')

✓ Mechanism map loaded: 31 clusters
  Sample (wound_healing): ['anti-inflammatory', 'antioxidant', 'collagen synthesis', 'angiogenesis', 'cell proliferation', 're-epithelialization', 'extracellular matrix remodeling', 'antimicrobial']

disease_synonyms type: <class 'list'>
claim_ids_covered type: <class 'list'>
Sample disease_synonyms (first 3): ['pruritus', 'hyperhidrosis', 'warts']
Sample claim_ids_covered: ['CLAIM_00001']
✓ other_known_names parsed
  Sample (entry_id=1): ['Atrilal', 'Bullwort', 'Greater Ammi']

Search units ready: 5014
Columns: ['search_unit_id', 'entry_id', 'scientific_name', 'gbif_validated', 'other_known_names', 'cluster_label', 'broader_category', 'disease_synonyms', 'claim_ids_covered', 'claims_count', 'other_names_list']


In [5]:
# ── Cell 3 — Query Builder Function ───────────────────────────────────────

def select_best_terms(terms, max_terms=12):
    """
    From a list of disease/mechanism terms, select the most
    PubMed-friendly ones by preferring shorter cleaner terms.
    Removes duplicates and overly long compound expressions.
    """
    # Deduplicate while preserving order
    seen = set()
    unique_terms = []
    for t in terms:
        t_clean = t.strip().lower()
        if t_clean not in seen:
            seen.add(t_clean)
            unique_terms.append(t.strip())

    # Sort by word count ascending (shorter = cleaner PubMed term)
    unique_terms.sort(key=lambda t: len(t.split()))

    # Remove terms longer than 5 words (historical compound expressions)
    unique_terms = [t for t in unique_terms if len(t.split()) <= 5]

    return unique_terms[:max_terms]


def build_pubmed_query(herb_name, terms):
    selected = select_best_terms(terms)
    if not selected:
        return f'"{herb_name}"'
    terms_joined = ' OR '.join([f'"{t}"' for t in selected])
    return f'"{herb_name}" AND ({terms_joined})'


def build_europepmc_query(herb_name, terms):
    """
    Build a Europe PMC Boolean query.
    "Herb name" AND ("term1" OR "term2")
    """
    selected = select_best_terms(terms)
    if not selected:
        return f'"{herb_name}"'

    terms_joined = ' OR '.join([f'"{t}"' for t in selected])
    return f'"{herb_name}" AND ({terms_joined})'


def build_semantic_scholar_query(herb_name, terms):
    """
    Build a Semantic Scholar keyword query.
    No Boolean operators — just space-separated keywords.
    """
    selected = select_best_terms(terms, max_terms=5)
    keywords = [herb_name] + selected
    return ' '.join(keywords)


# ── Test with a real search unit ───────────────────────────────────────────
sample = search_units[search_units['cluster_label'] == 'wound_healing'].iloc[0]
sci_name = sample['scientific_name']
synonyms = sample['disease_synonyms']
mechanisms = mechanism_map['wound_healing']

print(f'=== Test Search Unit ===')
print(f'Scientific name: {sci_name}')
print(f'Cluster: wound_healing')
print(f'Total synonyms available: {len(synonyms)}')
print()

print('── Primary query (synonyms) ──')
print(f'PubMed:           {build_pubmed_query(sci_name, synonyms)}')
print(f'Europe PMC:       {build_europepmc_query(sci_name, synonyms)}')
print(f'Semantic Scholar: {build_semantic_scholar_query(sci_name, synonyms)}')
print()

print('── Fallback query (mechanism terms) ──')
print(f'PubMed:           {build_pubmed_query(sci_name, mechanisms)}')
print(f'Europe PMC:       {build_europepmc_query(sci_name, mechanisms)}')
print(f'Semantic Scholar: {build_semantic_scholar_query(sci_name, mechanisms)}')
print()

print('── Fallback query (other known names) ──')
other_names = sample['other_names_list']
if other_names:
    print(f'Other names available: {other_names}')
    combined_name = ' OR '.join([f'"{n}"' for n in other_names[:3]])
    print(f'PubMed: ({combined_name})[Title/Abstract] AND ({build_europepmc_query("", synonyms).split(" AND ")[1]})')
else:
    print('No other names available for this unit')

=== Test Search Unit ===
Scientific name: Ammi majus
Cluster: wound_healing
Total synonyms available: 105

── Primary query (synonyms) ──
PubMed:           "Ammi majus" AND ("ulcers" OR "wounds" OR "burns" OR "fistulae" OR "abrasions" OR "fistula" OR "gangrene" OR "hematoma" OR "ulcerations" OR "internal ulcers" OR "phagedenic ulcers" OR "dog bite")
Europe PMC:       "Ammi majus" AND ("ulcers" OR "wounds" OR "burns" OR "fistulae" OR "abrasions" OR "fistula" OR "gangrene" OR "hematoma" OR "ulcerations" OR "internal ulcers" OR "phagedenic ulcers" OR "dog bite")
Semantic Scholar: Ammi majus ulcers wounds burns fistulae abrasions

── Fallback query (mechanism terms) ──
PubMed:           "Ammi majus" AND ("anti-inflammatory" OR "antioxidant" OR "angiogenesis" OR "re-epithelialization" OR "antimicrobial" OR "collagen synthesis" OR "cell proliferation" OR "extracellular matrix remodeling")
Europe PMC:       "Ammi majus" AND ("anti-inflammatory" OR "antioxidant" OR "angiogenesis" OR "re-epithe

In [6]:
def get_other_names(other_names_list, max_names=5):
    """Return other known names as-is, just cap the count."""
    return [n for n in other_names_list if len(n.strip()) > 3][:max_names]

In [16]:
!pip install biopython -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.2 MB/s eta 0:00:00


In [7]:
# ── Cell 4 — API Clients Setup ─────────────────────────────────────────────

!pip install biopython -q

import requests
import time
from Bio import Entrez

# ── PubMed (Entrez) ────────────────────────────────────────────────────────
Entrez.email = "22-101172@students.eui.edu.eg"
Entrez.api_key = "9b29537a988725f3edcbb4bbd44b01796308"
PUBMED_RATE_LIMIT = 0.11

# ── Europe PMC ────────────────────────────────────────────────────────────
EUROPEPMC_BASE_URL = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
EUROPEPMC_RATE_LIMIT = 0.5

# ── Test connections ───────────────────────────────────────────────────────
def test_pubmed():
    try:
        handle = Entrez.esearch(db="pubmed", term="Ammi majus", retmax=1)
        record = Entrez.read(handle)
        handle.close()
        return int(record["Count"]) > 0
    except Exception as e:
        print(f"PubMed error: {e}")
        return False

def test_europepmc():
    try:
        params = {"query": "Ammi majus", "format": "json", "pageSize": 1}
        response = requests.get(EUROPEPMC_BASE_URL, params=params, timeout=10)
        data = response.json()
        return data['hitCount'] > 0
    except Exception as e:
        print(f"Europe PMC error: {e}")
        return False

print("Testing API connections...")
print()

pubmed_ok = test_pubmed()
print(f"PubMed:     {'✓ Connected' if pubmed_ok else '✗ Failed'}")
time.sleep(0.5)

europepmc_ok = test_europepmc()
print(f"Europe PMC: {'✓ Connected' if europepmc_ok else '✗ Failed'}")
print()

if pubmed_ok and europepmc_ok:
    print("✓ Both APIs connected — ready to search")
elif pubmed_ok:
    print("⚠ PubMed only — Europe PMC failed")
else:
    print("✗ PubMed failed — do not proceed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.8 MB/s eta 0:00:00
Testing API connections...

PubMed:     ✓ Connected
Europe PMC: ✓ Connected

✓ Both APIs connected — ready to search


In [8]:
# ── Cell 5 — Relevance Filter Function ────────────────────────────────────

def is_relevant_paper(title, abstract, scientific_name, other_names, terms):
    """
    Check if a paper is relevant to this search unit.
    Paper must mention:
      1. The herb (scientific name OR any other known name)
      2. At least one disease/mechanism term
    Check is case-insensitive across title + abstract combined.
    """
    # Combine title and abstract into one searchable text
    text = f"{title} {abstract}".lower()

    # Check herb presence
    herb_names = [scientific_name] + other_names
    herb_found = any(name.lower() in text for name in herb_names)

    if not herb_found:
        return False

    # Check at least one disease/mechanism term is present
    term_found = any(term.lower() in text for term in terms)

    return term_found


# ── Test with a real example ───────────────────────────────────────────────
test_title = "Ammi majus and wound healing properties in vitro"
test_abstract = "This study investigates the effect of Ammi majus extract on collagen synthesis and tissue repair in dermal fibroblasts."

sample_unit = search_units.iloc[0]
sci_name = sample_unit['scientific_name']
other_names = sample_unit['other_names_list']
synonyms = sample_unit['disease_synonyms']
mechanisms = mechanism_map[sample_unit['cluster_label']]

print("=== Relevance Filter Test ===")
print(f"Title: {test_title}")
print(f"Abstract: {test_abstract[:80]}...")
print(f"Scientific name: {sci_name}")
print(f"Terms used (first 5): {synonyms[:5]}")
print()

result = is_relevant_paper(test_title, test_abstract, sci_name, other_names, synonyms + mechanisms)
print(f"Relevant: {result}")
print()

# Test a non-relevant paper
test_title_bad = "Effects of Nigella sativa on blood pressure"
test_abstract_bad = "This study examines Nigella sativa extract in hypertensive patients."

result_bad = is_relevant_paper(test_title_bad, test_abstract_bad, sci_name, other_names, synonyms + mechanisms)
print(f"Non-relevant paper correctly rejected: {not result_bad}")

=== Relevance Filter Test ===
Title: Ammi majus and wound healing properties in vitro
Abstract: This study investigates the effect of Ammi majus extract on collagen synthesis a...
Scientific name: Ammi majus
Terms used (first 5): ['pruritus', 'hyperhidrosis', 'warts', 'vitiligo', 'melasma']

Relevant: True

Non-relevant paper correctly rejected: True


In [16]:
# # ── Cell 6 — Cache Setup ───────────────────────────────────────────────────

import json
import os

CACHE_FILE = 'search_cache.json'

def load_cache():
    """Load existing cache from disk. Returns empty dict if no cache exists."""
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, 'r') as f:
            cache = json.load(f)
        print(f'✓ Existing cache loaded: {len(cache)} search units already completed')
        return cache
    else:
        print('✓ No existing cache — starting fresh')
        return {}

def save_to_cache(cache, search_unit_id, result):
    """Save one search unit result to cache immediately after retrieval."""
    cache[search_unit_id] = result
    with open(CACHE_FILE, 'w') as f:
        json.dump(cache, f)

def is_cached(cache, search_unit_id):
    """Check if a search unit has already been searched."""
    return search_unit_id in cache

# ── Initialize cache ───────────────────────────────────────────────────────
cache = load_cache()

remaining = len(search_units) - len(cache)
print(f'Search units remaining: {remaining} / {len(search_units)}')

✓ Existing cache loaded: 500 search units already completed
Search units remaining: 4514 / 5014


In [18]:
# ── Cell 7 — Search Functions ──────────────────────────────────────────────

from Bio import Medline

# ── PubMed Search ──────────────────────────────────────────────────────────

def search_pubmed(query, max_results=5):
    """
    Search PubMed with a Boolean query.
    Returns list of paper dicts with all required fields.
    """
    try:
        time.sleep(PUBMED_RATE_LIMIT)

        # Step 1: Search — get PMIDs
        handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
        record = Entrez.read(handle)
        handle.close()

        pmids = record["IdList"]
        if not pmids:
            return []

        time.sleep(PUBMED_RATE_LIMIT)

        # Step 2: Fetch full records
        handle = Entrez.efetch(
            db="pubmed",
            id=",".join(pmids),
            rettype="medline",
            retmode="text"
        )
        records = list(Medline.parse(handle))
        handle.close()

        papers = []
        for r in records:
            abstract = r.get("AB", "")
            title = r.get("TI", "")
            if not title:
                continue

            papers.append({
                "paper_title":     title,
                "abstract":        abstract,
                "pmid":            r.get("PMID", ""),
                "doi":             next((aid for aid in r.get("AID", []) if "[doi]" in aid), "").replace(" [doi]", ""),
                "year":            r.get("DP", "")[:4] if r.get("DP") else "",
                "journal_or_venue":r.get("JT", ""),
                "study_type":      ", ".join(r.get("PT", [])),
                "source":          "PubMed"
            })

        return papers

    except Exception as e:
        print(f"  PubMed error: {e}")
        return []


# ── Europe PMC Search ──────────────────────────────────────────────────────

def search_europepmc(query, max_results=5):
    """
    Search Europe PMC with a Boolean query.
    Returns list of paper dicts with all required fields.
    """
    try:
        time.sleep(EUROPEPMC_RATE_LIMIT)

        params = {
            "query":    query,
            "format":   "json",
            "pageSize": max_results,
            "resultType": "core"
        }

        response = requests.get(EUROPEPMC_BASE_URL, params=params, timeout=30)
        data = response.json()

        results = data.get("resultList", {}).get("result", [])
        if not results:
            return []

        papers = []
        for r in results:
            title    = r.get("title", "")
            abstract = r.get("abstractText", "")
            if not title:
                continue

            papers.append({
                "paper_title":     title,
                "abstract":        abstract,
                "pmid":            str(r.get("pmid", "")),
                "doi":             r.get("doi", ""),
                "year":            str(r.get("pubYear", "")),
                "journal_or_venue": r.get("journalInfo", {}).get("journal", {}).get("title", ""),
                "study_type":      r.get("pubType", ""),
                "source":          "Europe PMC",
                "is_open_access": r.get("isOpenAccess", "N"),
                "cited_by_count":  r.get("citedByCount", 0),
            })

        return papers

    except Exception as e:
        print(f"  Europe PMC error: {e}")
        return []


# ── Test both functions ────────────────────────────────────────────────────
print("=== Testing search functions ===")
print()

test_query_pubmed    = build_pubmed_query("Ammi majus", ["vitiligo", "skin pigmentation", "leukoderma", "psoralen"])
test_query_europepmc = build_europepmc_query("Ammi majus", ["vitiligo", "skin pigmentation", "leukoderma", "psoralen"])

print(f"PubMed query:     {test_query_pubmed}")
print(f"Europe PMC query: {test_query_europepmc}")
print()

pubmed_results = search_pubmed(test_query_pubmed, max_results=3)
print(f"PubMed results: {len(pubmed_results)} papers")
for p in pubmed_results:
    print(f"  - [{p['source']}] {p['paper_title'][:70]}")
    print(f"    PMID: {p['pmid']} | Year: {p['year']} | Study type: {p['study_type'][:40]}")
print()

europepmc_results = search_europepmc(test_query_europepmc, max_results=3)
print(f"Europe PMC results: {len(europepmc_results)} papers")
for p in europepmc_results:
    print(f"  - [{p['source']}] {p['paper_title'][:70]}")
    print(f"    PMID: {p['pmid']} | Year: {p['year']} | Journal: {p['journal_or_venue'][:40]}")

=== Testing search functions ===

PubMed query:     "Ammi majus" AND ("vitiligo" OR "leukoderma" OR "psoralen" OR "skin pigmentation")
Europe PMC query: "Ammi majus" AND ("vitiligo" OR "leukoderma" OR "psoralen" OR "skin pigmentation")

PubMed results: 3 papers
  - [PubMed] Integrated Metabolome and Transcriptome Analysis in Ammi majus L. Reve
    PMID: 40272701 | Year: 2026 | Study type: Journal Article
  - [PubMed] Exogenous application of salicylic acid and putrescine triggers physio
    PMID: 39089592 | Year: 2024 | Study type: Journal Article
  - [PubMed] Ultra-Performance Liquid Chromatography Coupled with Mass Metabolic Pr
    PMID: 37887369 | Year: 2023 | Study type: Journal Article

Europe PMC results: 3 papers
  - [Europe PMC] Integrated Metabolome and Transcriptome Analysis in Ammi majus L. Reve
    PMID: 40272701 | Year: 2026 | Journal: Biochemical genetics
  - [Europe PMC] The role of phototherapy in pediatric dermatology.
    PMID: 41483505 | Year: 2026 | Journal: Anais b

In [ ]:
# ── Cell 8 — Main Search Loop ──────────────────────────────────────────────

from datetime import datetime

results_list = []   # all retrieved papers
search_log   = []   # every query executed
MIN_PAPERS   = 3    # target papers per search unit

def search_one_unit(row):
    """
    Execute full cascade for one search unit.
    Returns (papers_found, log_entries)
    """
    su_id      = row['search_unit_id']
    sci_name   = row['scientific_name']
    other_names= row['other_names_list']
    synonyms   = row['disease_synonyms']
    cluster    = row['cluster_label']
    mechanisms = mechanism_map.get(cluster, [])
    all_terms  = synonyms + mechanisms

    papers_found = []
    log_entries  = []

    def run_search(herb_name, terms, cascade_level, source_label):
        """Run one query against both APIs, return relevant papers."""
        found = []

        # PubMed
        pubmed_query = build_pubmed_query(herb_name, terms)
        pubmed_papers = search_pubmed(pubmed_query, max_results=5)
        relevant_pubmed = [
            p for p in pubmed_papers
            if is_relevant_paper(p['paper_title'], p['abstract'],
                                 sci_name, other_names, all_terms)
        ]
        log_entries.append({
            'search_unit_id':   su_id,
            'query':            pubmed_query,
            'source':           'PubMed',
            'cascade_level':    cascade_level,
            'results_returned': len(pubmed_papers),
            'results_kept':     len(relevant_pubmed),
            'timestamp':        datetime.now().isoformat()
        })
        found.extend(relevant_pubmed)

        # Buffer before Europe PMC
        time.sleep(0.3)  # ← add this

        # Europe PMC — only if PubMed didn't give enough
        if len(found) < MIN_PAPERS:
            epmc_query  = build_europepmc_query(herb_name, terms)
            epmc_papers = search_europepmc(epmc_query, max_results=5)
            relevant_epmc = [
                p for p in epmc_papers
                if is_relevant_paper(p['paper_title'], p['abstract'],
                                     sci_name, other_names, all_terms)
                and p['pmid'] not in {x['pmid'] for x in found}
            ]
            log_entries.append({
                'search_unit_id':   su_id,
                'query':            epmc_query,
                'source':           'Europe PMC',
                'cascade_level':    cascade_level,
                'results_returned': len(epmc_papers),
                'results_kept':     len(relevant_epmc),
                'timestamp':        datetime.now().isoformat()
            })
            found.extend(relevant_epmc)

        return found

    # ── Level 1: scientific name + disease synonyms ────────────────────────
    papers = run_search(sci_name, synonyms, cascade_level=1, source_label='primary')
    papers_found.extend(papers)

    # ── Level 2: scientific name + mechanism terms (fallback) ──────────────
    if len(papers_found) < MIN_PAPERS and mechanisms:
        papers = run_search(sci_name, mechanisms, cascade_level=2, source_label='mechanism_fallback')
        new_papers = [p for p in papers if p['pmid'] not in {x['pmid'] for x in papers_found}]
        papers_found.extend(new_papers)

    # ── Level 3: other known names + disease synonyms (fallback) ──────────
    if len(papers_found) < MIN_PAPERS and other_names:
        usable_names = get_other_names(other_names, max_names=3)
        if usable_names:
            for name in usable_names:
                if len(papers_found) >= MIN_PAPERS:
                    break
                papers = run_search(name, synonyms, cascade_level=3, source_label='other_names_fallback')
                new_papers = [p for p in papers if p['pmid'] not in {x['pmid'] for x in papers_found}]
                papers_found.extend(new_papers)

    # ── Cap at 3 papers ────────────────────────────────────────────────────
    papers_found = papers_found[:3]

    return papers_found, log_entries


# ── Run the main loop ──────────────────────────────────────────────────────
print(f"Starting search for {len(search_units)} search units...")
print(f"Already cached: {len(cache)}")
print(f"Remaining: {len(search_units) - len(cache)}")
print()

for idx, row in search_units.iterrows():
    su_id = row['search_unit_id']

    # Skip if already cached
    if is_cached(cache, su_id):
        continue

    # Run cascade
    papers, logs = search_one_unit(row)

    # Attach search_unit_id to each paper
    for p in papers:
        p['search_unit_id'] = su_id

    # Build cache entry
    cache_entry = {
        'status':        'found' if papers else 'no_evidence',
        'papers':        papers,
        'cascade_level': logs[-1]['cascade_level'] if logs else None
    }

    # Save to cache immediately
    save_to_cache(cache, su_id, cache_entry)

    # Backup to Google Drive every 50 units ← ADD THIS
    if len(cache) % 50 == 0:
      import shutil
      shutil.copy('search_cache.json',
                '/content/drive/MyDrive/pipeline_cache/search_cache.json')

    # Collect results and logs
    if papers:
        results_list.extend(papers)
    else:
        results_list.append({
            'search_unit_id': su_id,
            'paper_title':    None,
            'abstract':       None,
            'pmid':           None,
            'doi':            None,
            'year':           None,
            'journal_or_venue': None,
            'study_type':     None,
            'source':         None,
            'status':         'No Evidence Found'
        })
    search_log.extend(logs)

    # Progress report every 50 units
    completed = len(cache)
    if completed % 50 == 0:
        found_count = sum(1 for v in cache.values() if v['status'] == 'found')
        print(f"[{completed}/{len(search_units)}] Found evidence: {found_count} | No evidence: {completed - found_count}")

# ── Final summary ──────────────────────────────────────────────────────────
print()
print("═══ SEARCH COMPLETE ═══")
total        = len(cache)
found        = sum(1 for v in cache.values() if v['status'] == 'found')
no_evidence  = total - found
print(f"Total units searched:    {total}")
print(f"Units with evidence:     {found}")
print(f"Units with no evidence:  {no_evidence}")
print(f"Total papers collected:  {len(results_list)}")
print(f"Total queries logged:    {len(search_log)}")

Starting search for 5014 search units...
Already cached: 511
Remaining: 4503

  Europe PMC error: HTTPSConnectionPool(host='www.ebi.ac.uk', port=443): Read timed out. (read timeout=30)
  Europe PMC error: HTTPSConnectionPool(host='www.ebi.ac.uk', port=443): Read timed out. (read timeout=30)
[550/5014] Found evidence: 511 | No evidence: 39
[600/5014] Found evidence: 554 | No evidence: 46
[650/5014] Found evidence: 603 | No evidence: 47
[700/5014] Found evidence: 646 | No evidence: 54
  PubMed error: Search Backend failed: An error occurred while processing request. Status: 500. Source: /api/search/?r= Details: Search is temporarily unavailable. Please try again later. Details: Cannot connect to SOLR : null
[750/5014] Found evidence: 694 | No evidence: 56
  PubMed error: Search Backend failed: XML Exception at /pubmed_gen/rbuild/version/20260413/entrez/2.20/src/internal/txxmldoc/XmlDocument.cpp(1449): code=6, XML: Parse failed
  PubMed error: Search Backend failed: XML Exception at /pubm

#Case 1 (all in colab)

In [ ]:
# ── Cell 9 — Build Final Output Files (in-memory) ─────────────────────────

import pandas as pd

evidence_rows = []
for su_id, entry in cache.items():
    if entry['status'] == 'found':
        for paper in entry['papers']:
            paper['search_unit_id'] = su_id
            paper['status'] = 'found'
            evidence_rows.append(paper)
    else:
        evidence_rows.append({
            'search_unit_id':   su_id,
            'paper_title':      None,
            'abstract':         None,
            'pmid':             None,
            'doi':              None,
            'year':             None,
            'journal_or_venue': None,
            'study_type':       None,
            'source':           None,
            'status':           'No Evidence Found'
        })

evidence_df = pd.DataFrame(evidence_rows)

# claims_ref already in memory from Cell 1A upload
# No need to re-read from disk

print('=== VALIDATION ===')
print(f'Search units in cache:        {len(cache)}')
print(f'Search units expected:        {len(search_units)}')
print(f'Evidence rows total:          {len(evidence_df)}')
print(f'Papers found:                 {len(evidence_df[evidence_df["status"]=="found"])}')
print(f'No Evidence Found units:      {len(evidence_df[evidence_df["status"]=="No Evidence Found"])}')
print()

covered = set(cache.keys())
claims_su_ids = set(claims_ref['search_unit_id'].unique())
missing = claims_su_ids - covered
print(f'Search units in claims not in cache: {len(missing)}')
if not missing:
    print('✓ All claim search units covered')

evidence_df.to_excel('evidence_table_full.xlsx', index=False)
print(f'\n✓ Saved evidence_table_full.xlsx')

In [ ]:
# ── Cell 10 — Download (in-memory) ────────────────────────────────────────

from google.colab import files

# Final Drive backup
import shutil
shutil.copy('search_cache.json',
            '/content/drive/MyDrive/pipeline_cache/search_cache.json')
print('✓ Final cache backed up to Drive')

# Download evidence table
files.download('evidence_table_full.xlsx')
print('✓ evidence_table_full.xlsx downloaded')

# Build and download search log
search_log_df = pd.DataFrame(search_log)
search_log_df.to_excel('search_log.xlsx', index=False)
files.download('search_log.xlsx')
print('✓ search_log.xlsx downloaded')

#Case 2 (from drive)

In [ ]:
# ── Cell 9 — Build Final Output Files (from Drive) ────────────────────────

import pandas as pd
import json
from google.colab import drive
from google.colab import files

drive.mount('/content/drive')

# Load cache from Drive
CACHE_FILE = '/content/drive/MyDrive/pipeline_cache/search_cache.json'
with open(CACHE_FILE, 'r') as f:
    cache = json.load(f)
print(f'✓ Cache loaded from Drive: {len(cache)} units')

# Upload claims_reference when prompted
uploaded = files.upload()
claims_ref = pd.read_excel('claims_reference.xlsx')

# Build evidence rows
evidence_rows = []
for su_id, entry in cache.items():
    if entry['status'] == 'found':
        for paper in entry['papers']:
            paper['search_unit_id'] = su_id
            paper['status'] = 'found'
            evidence_rows.append(paper)
    else:
        evidence_rows.append({
            'search_unit_id':   su_id,
            'paper_title':      None,
            'abstract':         None,
            'pmid':             None,
            'doi':              None,
            'year':             None,
            'journal_or_venue': None,
            'study_type':       None,
            'source':           None,
            'status':           'No Evidence Found'
        })

evidence_df = pd.DataFrame(evidence_rows)

print('=== VALIDATION ===')
print(f'Search units in cache:        {len(cache)}')
print(f'Evidence rows total:          {len(evidence_df)}')
print(f'Papers found:                 {len(evidence_df[evidence_df["status"]=="found"])}')
print(f'No Evidence Found units:      {len(evidence_df[evidence_df["status"]=="No Evidence Found"])}')
print()

covered = set(cache.keys())
claims_su_ids = set(claims_ref['search_unit_id'].unique())
missing = claims_su_ids - covered
print(f'Search units in claims not in cache: {len(missing)}')
if not missing:
    print('✓ All claim search units covered')

evidence_df.to_excel('evidence_table_full.xlsx', index=False)
print(f'\n✓ Saved evidence_table_full.xlsx')

In [ ]:
# ── Cell 10 — Download (from Drive) ───────────────────────────────────────

from google.colab import files

files.download('evidence_table_full.xlsx')
print('✓ evidence_table_full.xlsx downloaded')

# Search log won't be available after disconnect
# so we skip it and note it as unavailable
print('⚠ Search log not available after disconnect — only evidence table exported')

In [14]:
# ── Export first 500 units for teammate ───────────────────────────────────

import pandas as pd

# Build evidence table from cache
evidence_rows = []
for su_id, entry in cache.items():
    if entry['status'] == 'found':
        for paper in entry['papers']:
            paper['search_unit_id'] = su_id
            evidence_rows.append(paper)
    else:
        evidence_rows.append({
            'search_unit_id': su_id,
            'paper_title': None,
            'abstract': None,
            'pmid': None,
            'doi': None,
            'year': None,
            'journal_or_venue': None,
            'study_type': None,
            'source': None,
            'status': 'No Evidence Found'
        })

evidence_df = pd.DataFrame(evidence_rows)

# Get matching claims
covered_su_ids = set(cache.keys())
claims_ref = pd.read_excel('claims_reference.xlsx')
claims_subset = claims_ref[claims_ref['search_unit_id'].isin(covered_su_ids)]

# Save
evidence_df.to_excel('evidence_table_500.xlsx', index=False)
claims_subset.to_excel('claims_reference_500.xlsx', index=False)

from google.colab import files
files.download('evidence_table_500.xlsx')
files.download('claims_reference_500.xlsx')

print(f'Evidence rows: {len(evidence_df)}')
print(f'Claims rows: {len(claims_subset)}')
print(f'Search units covered: {len(covered_su_ids)}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Evidence rows: 1322
Claims rows: 718
Search units covered: 500
